Code for analyzing the 700 hPa geopotential height and wind fields from CM2.5-FLOR Preindustrial simulations forced with CTRL and modified topographic boundary conditions and comparing model output to MERRA-2 reanalysis.

Relevant for manuscript Figure 4 (?)

In [1]:
import os
import sys
# block warnings from printing
import warnings
warnings.filterwarnings('ignore')
warnings.simplefilter('ignore')

import collections
import pandas as pd
import xarray as xr
import cf_xarray as cf
xr.set_options(keep_attrs=True)
import netCDF4 as nc
import numpy as np
np.seterr(divide='ignore', invalid='ignore')
import metpy.calc as mp
from metpy.units import units
import iris
from windspharm.iris import VectorWind
import windspharm.xarray as wph
from scipy.stats import ttest_ind, ttest_rel
from datetime import datetime

import cartopy
cartopy.config['data_dir'] = "/discover/nobackup/projects/jh_tutorials/JH_examples/JH_datafiles/Cartopy"
cartopy.config['pre_existing_data_dir'] = "/discover/nobackup/projects/jh_tutorials/JH_examples/JH_datafiles/Cartopy"
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from cartopy.mpl.gridliner import LONGITUDE_FORMATTER, LATITUDE_FORMATTER
from shapely.geometry.polygon import LinearRing

import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from matplotlib.offsetbox import AnchoredText
import matplotlib.gridspec as gridspec
import matplotlib.ticker as mticker
import matplotlib.colors as mcolors
from matplotlib.colors import TwoSlopeNorm
from matplotlib import cm
from matplotlib.colors import ListedColormap,LinearSegmentedColormap
import cmocean.cm as cmo
import seaborn as sns

# settings
%config InlineBackend.figure_format = 'retina'

# add path to custom functions
module_path = os.path.abspath(os.path.join('..'))
if module_path not in sys.path:
    sys.path.append(module_path+"/py_functions")
from map_plot_tools import *
from colorbar_funcs import *
from data_funcs import *
from stats_funcs import *

In [2]:
def windSpd(u,v):
   windSpd=np.sqrt(u**2 + v**2)
   return windSpd

In [3]:
### +++ DATA PATHS +++ ###
dpath0 = '/discover/nobackup/projects/giss/baldwin_nip/dmkumar'
opath  = '/home/dmkumar/JupyterLinks/notebooks/topo_nam/figs'

flor_runs = ['ctrl', 'hitopo', 'hicam']
varns     = ['u700', 'v700', 'h700']

files = {
    'merra2': {},
    'ctrl':   {},
    'hitopo': {},
    'hicam':  {},
    'topo':   {}
}

# MERRA-2 Reanalysis
files['merra2']['u700'] = f'{dpath0}/obs_data/merra2/merra2.U.1980-2022.monthly.nc' 
files['merra2']['v700'] = f'{dpath0}/obs_data/merra2/merra2.V.1980-2022.monthly.nc' 
files['merra2']['h700'] = f'{dpath0}/obs_data/merra2/merra2.H.1980-2022.monthly.nc'

# CM2.5-FLOR Runs
for run in flor_runs:
    for varn in varns:
        files[run][varn] = f'{dpath0}/FLOR/{run}/pi/flor.{run}.{varn}.monthly.nc'

# Obs & Model Topography
files['topo']['etopo']  = f'{dpath0}/topo_files/obs.etopo5.zsurf.nc'
files['topo']['ctrl']   = f'{dpath0}/topo_files/flor.ctrl.zsurf.nc'
files['topo']['hitopo'] = f'{dpath0}/topo_files/flor.hitopo.zsurf.nc'
files['topo']['hicam']  = f'{dpath0}/topo_files/flor.hicam.zsurf.nc'

In [ ]:
### +++ LOAD DATA +++ ###

# lat lon bounds
latmin = 10
latmax = 70
lonmin = 200
lonmax = 300
# time bounds
n_keep_years = -100 # last 100 years
n_keep_mons  = n_keep_years * 12

# initialize dictionaries
dat = {
    'merra2': {},
    'ctrl':   {},
    'hitopo': {},
    'hicam':  {},
}
topo = {}

# --- Read in raw fields --- #
print('Loading data...')

# MERRA-2: select 700 hPa level
print('-> MERRA-2')
for varn, mvarn in zip(varns, ['U','V','H']):
    ds = (
        xr.open_dataset(files['merra2'][varn])
        [mvarn]
        .sel(lev=700, drop=True)
    )
    dat['merra2'][varn] = (
        lonFlip(ds)
        #.sel(lat=slice(latmin,latmax), lon=slice(lonmin,lonmax))
    )
    del ds

# CM2.5-FLOR PI: last n_years of simulation
print('-> CM2.5-FLOR')
for run in flor_runs:
    for varn in varns:
        dat[run][varn] = (
            xr.open_dataset(files[run][varn])
            [varn]
            [n_keep_mons:]
            .rename({'grid_xt': 'lon', 'grid_yt': 'lat'})
            #.sel(lat=slice(latmin, latmax), lon=slice(lonmin, lonmax))
        )

# --- Calculate speed of 700 hPa winds --- #
print('\nCalculating wind speed...')
for key in dat.keys():
    dat[key]['spd700'] = windSpd(dat[key]['u700'], dat[key]['v700'])
    
# --- Calculate stationary wave pattern in 700 hPa geopotential heights --- #
print('\nCalculating stationary wave patterns...')
for key in dat.keys():
    dat[key]['swp700'] = dat[key]['h700'] - (dat[key]['h700'].mean(dim='lon'))

# --- Calculate streamfunction of 700 hPa wind components --- #
print('\nCalculating streamfunctions...')
for key in dat.keys():
    dat[key]['psi700'] = (
        wph.VectorWind(dat[key]['u700'], dat[key]['v700'])
        .streamfunction()
    )

# --- Read in surface topography fields --- #
print('\nLoading topography fields...')
ds = xr.open_dataset(files['topo']['etopo'])['ROSE'].rename({'ETOPO05_X': 'lon', 'ETOPO05_Y': 'lat'})
topo['etopo'] = (
    ds
    .where(ds > 0, np.nan)
    #.sel(lat=slice(latmin,latmax), lon=slice(lonmin,lonmax))
); del ds
for run in flor_runs:
    topo[run] = (
        xr.open_dataset(files['topo'][run]).ZSURF
        .rename({'GRID_XT': 'lon', 'GRID_YT': 'lat'})
        #.sel(lat=slice(latmin,latmax), lon=slice(lonmin,lonmax))
    )

print('\nDone.')

Loading data...
-> MERRA-2
-> CM2.5-FLOR

Calculating wind speed...

Calculating stationary wave patterns...


In [ ]:
### +++ JAS SEASONAL MEANS +++ ###
season = 'JAS'
mons   = [7, 8, 9]
varns  = ['u700', 'v700', 'h700', 'spd700', 'swp700', 'psi700']

jas_mean = {
    'merra2': {},
    'ctrl':   {},
    'hitopo': {},
    'hicam':  {},
}
ann_jas_mean = {
    'merra2': {},
    'ctrl':   {},
    'hitopo': {},
    'hicam':  {},
}

print('Calculating JAS means...')
for key in dat.keys():
    for varn in varns:
        da = dat[key][varn]
        custom_seasons = xr.where(da['time'].dt.month.isin(mons), season, 'Other')
        jas_mean[key][varn]     = da.groupby(custom_seasons).mean('time').rename({'month': 'season'}).sel(season=season)
        ann_jas_mean[key][varn] = da.sel(time=da['time'].dt.month.isin(mons)).groupby('time.year').mean(dim='time')
print('Done.')

In [ ]:
### +++ MODEL BIAS (MODEL - OBS) +++ ###

# --- Regrid obs to model grid ---
print('Re-gridding obs to model grid...\n')

jas_mean_regrid     = {'obs_flor': {}}
ann_jas_mean_regrid = {'obs_flor': {}}

lats = dat['ctrl']['h700'].lat
lons = dat['ctrl']['h700'].lon
for varn in varns:
    jas_mean_regrid['obs_flor'][varn]     = jas_mean['merra2'][varn].interp(lat=lats, lon=lons, method='linear')
    ann_jas_mean_regrid['obs_flor'][varn] = ann_jas_mean['merra2'][varn].interp(lat=lats, lon=lons, method='linear')

# --- Significance testing of fields ---
print('Significance testing (model - obs)...\n')

obs_diff      = {'ctrl': {}, 'hitopo': {}, 'hicam': {}}
obs_diff_mask = {'ctrl': {}, 'hitopo': {}, 'hicam': {}}
obs_ptvals    = {'ctrl': {}, 'hitopo': {}, 'hicam': {}}

for run in flor_runs:
    print(f'\n-> {run}')
    for varn in varns:
        print(f'---> {varn}')
        diff_, mask_, pvals_ = sigtest2n(ann_jas_mean[run][varn], ann_jas_mean_regrid['obs_flor'][varn],
                                         jas_mean[run][varn],  jas_mean_regrid['obs_flor'][varn])
        obs_diff[run][varn]      = diff_
        obs_diff_mask[run][varn] = mask_
        obs_ptvals[run][varn]    = pvals_


# --- Mask vectors where both are significant ---
obs_usig = {'ctrl': {}, 'hitopo': {}, 'hicam': {}}
obs_vsig = {'ctrl': {}, 'hitopo': {}, 'hicam': {}}

for run in flor_runs:
    obs_usig[run] = np.where((obs_diff_mask[run]['u700'].mask==False) &
                             (obs_diff_mask[run]['v700'].mask==False),
                             obs_diff_mask[run]['u700'].data, np.nan)
    obs_vsig[run] = np.where((obs_diff_mask[run]['u700'].mask==False) &
                             (obs_diff_mask[run]['v700'].mask==False),
                             obs_diff_mask[run]['v700'].data, np.nan)

print('\nDone.')

In [ ]:
### +++ TOPO SENSITIVITY (Modified_Topo - CTRL) +++ ###

# --- Significance testing of fields ---
print('Significance testing (modified - ctrl)...')

model_diff      = {'hitopo':{}, 'hicam':{}}
model_diff_mask = {'hitopo':{}, 'hicam':{}}
model_ptvals    = {'hitopo':{}, 'hicam':{}}

for run in model_diff.keys():
    print(f'\n-> {run}')
    for varn in varns:
        print(f'---> {varn}')
        diff_, mask_, pvals_ = sigtest2n(ann_jas_mean[run][varn], ann_jas_mean['ctrl'][varn],
                                          jas_mean[run][varn], jas_mean['ctrl'][varn])
        model_diff[run][varn]      = diff_
        model_diff_mask[run][varn] = mask_
        model_ptvals[run][varn]    = pvals_

# --- Mask vectors where both are significant ---
model_usig = {'hitopo':{}, 'hicam':{}}
model_vsig = {'hitopo':{}, 'hicam':{}}

for run in ['hicam','hitopo']:
    model_usig[run] = np.where((model_diff_mask[run]['u700'].mask==False) &
                               (model_diff_mask[run]['v700'].mask==False),
                               model_diff_mask[run]['u700'].data, np.nan)
    model_vsig[run] = np.where((model_diff_mask[run]['u700'].mask==False) &
                               (model_diff_mask[run]['v700'].mask==False),
                               model_diff_mask[run]['v700'].data, np.nan)

print('\nDone.')

## FIGURES

### h700 climatology and stationary wave pattern

In [ ]:
# --- Settings ---
# plot specs
lw=1
text_kw={'color':'k', 'weight':'bold', 'size':20, 'ha':'center', 'va':'bottom'}
text_kw2={'color':'k', 'weight':'bold', 'size':26, 'ha':'center', 'va':'center'}
text_kw3={'color':'k', 'weight':'normal', 'size':14, 'ha':'left', 'va':'center'}
letters=['A','B','C','D','E','F','G','H','I','J','K','L','M','N']
tx=-105
ty=66
# topography contours
zlevels=np.linspace(1000,4000,4)
dzlevels=np.linspace(500,1500,5)
# h700 colormap
cmap=cm.RdYlBu_r
vmin=2900
vmax=3300
levels=np.linspace(vmin, vmax, 21)
norm=mpl.colors.BoundaryNorm(levels, cmap.N)
hlevels=np.arange(vmin-50, vmax+50, 25)
# swp700 colormap
dcmap=cm.RdBu_r
dlevels=np.linspace(-50, 50, 21)
dnorm=mpl.colors.BoundaryNorm(dlevels, dcmap.N)
# diff colormap
dcmap2=cmo.balance
dnorm2=mpl.colors.BoundaryNorm(np.linspace(-100, 100, 21), dcmap2.N)
dnorm3=mpl.colors.BoundaryNorm(np.linspace(-16, 16, 17), dcmap2.N)
# map specs
trans=ccrs.PlateCarree()
proj=ccrs.PlateCarree()
map_bnds=[225,290,10,65]
cols=[0,2]

fig, ax = plt.subplots(nrows=2, ncols=6, figsize=(35,10), layout='constrained', subplot_kw={'projection':proj})

for i,title in enumerate(['MERRA-2', 'CTRL', r'HI$\mathbf{_{gbl}}$', r'HI$\mathbf{_{mex}}$', r'HI$\mathbf{_{gbl}}$$-$CTRL', r'HI$\mathbf{_{mex}}$$-$CTRL']):
    ax[0,i].text(tx, ty, title, **text_kw)
    
#--- CLIMOS ---
for i,(key,topo_key) in enumerate(zip(jas_mean.keys(),topo.keys())):
    h700 = jas_mean[key]['h700']
    swp700 = jas_mean[key]['swp700']
    topo_ = topo[topo_key]
    cf=ax[0,i].pcolormesh(h700.lon, h700.lat, h700, cmap=cmap, norm=norm, transform=trans)
    cf2=ax[1,i].pcolormesh(swp700.lon, swp700.lat, swp700, cmap=cmap, norm=norm, transform=trans)

    ax[0,i].pcolormesh(topo_.lon, topo_.lat, topo_, levels=zlevels, linewidths=1, colors='black', transform=trans)
    ax[1,i].pcolormesh(topo_.lon, topo_.lat, topo_, levels=zlevels, linewidths=1, colors='black', transform=trans)

#--- FLOR CHANGE ---
ax[0,4].text(-165,40,' ')
for i,run in enumerate(['hitopo','hicam']):
    dh700 = model_diff[run]['h700']
    dswp700 = model_diff[run]['swp700']
    topo_ = topo[topo_key]
    cf3=ax[0,i+4].pcolormesh(dh700.lon, dh700.lat, dh700, cmap=dcmap2, norm=dnorm2, transform=trans)
    ax[0,i+4].contour(topo_.lon, topo_.lat, topo_-topo['ctrl'], levels=dzlevels, linewidths=1, colors='black', transform=trans)
    
    cf4=ax[1,i+4].pcolormesh(dswp700.lon, dswp700.lat, dswp700, cmap=dcmap2, norm=dnorm3, transform=trans)
    ax[1,i+4].contour(topo_.lon, topo_.lat, topo_-topo['ctrl'], levels=dzlevels, linewidths=1, colors='black', transform=trans)
    
for i, ax in enumerate(ax.flat): 
    # subplot labels
    ax.text(-136, ty+1, letters[i], **text_kw2)
    # map properties
    ax.coastlines(color='k', linewidth=1.5)
    ax.set_extent(map_bnds, crs=trans)
    gl=ax.gridlines(crs=trans, lw=.5, colors='black', alpha=1.0, linestyle='--', zorder=10, draw_labels=True)
    gl.bottom_labels=True; gl.left_labels=True; gl.top_labels=False; gl.right_labels=False
    gl.xformatter=LONGITUDE_FORMATTER
    gl.yformatter=LATITUDE_FORMATTER
    gl.xlabel_style={'color': 'black', 'weight': 'normal', 'size':14}
    gl.ylabel_style={'color': 'black', 'weight': 'normal', 'size':14}

# --- Colorbars --- 
# h500
cax=fig.add_axes([.64, .55, 0.0125, 0.4])
cbar=fig.colorbar(cf, ticks=np.linspace(2900,3300,9), orientation='vertical', extend='both', cax=cax) 
cbar.set_label('700 hPa Geopotential Height [m]', size=14, rotation=270, labelpad=20, ha='center')
cbar.ax.tick_params(labelsize=14)
# swp500
cax=fig.add_axes([.64, .05, 0.0125, 0.4])
cbar=fig.colorbar(cf2, ticks=np.linspace(-50,50,11), orientation='vertical', extend='both', cax=cax) 
cbar.set_label('700 hPa Stationary Wave Pattern [m]', size=14, rotation=270, labelpad=20, ha='center')
cbar.ax.tick_params(labelsize=14)
# h500 change
cax=fig.add_axes([1.005, .55, 0.0125, 0.4])
cbar=fig.colorbar(cf3, ticks=np.linspace(-100,100,11), orientation='vertical', extend='both', cax=cax) 
cbar.set_label('$\Delta$ 700 hPa Geopotential Height [m]', size=14, rotation=270, labelpad=20, ha='center')
cbar.ax.tick_params(labelsize=14)
# swp500 change
cax=fig.add_axes([1.005, .05, 0.0125, 0.4])                                 
cbar=fig.colorbar(cf4, ticks=np.linspace(-16,16,9), orientation='vertical', extend='both', cax=cax) 
cbar.set_label('$\Delta$ 700 hPa Stationary Wave Pattern [m]', size=14, rotation=270, labelpad=20, ha='center')
cbar.ax.tick_params(labelsize=14)

#plt.savefig(f'figs/h700.swp.flor.pdf', transparent=False, bbox_inches='tight')
#plt.savefig(f'figs/h700.swp.flor.png', transparent=False, bbox_inches='tight')

In [ ]:
# --- Settings ---
# plot specs
lw=1
text_kw={'color':'k', 'weight':'bold', 'size':20, 'ha':'center', 'va':'bottom'}
text_kw2={'color':'k', 'weight':'bold', 'size':26, 'ha':'center', 'va':'center'}
text_kw3={'color':'k', 'weight':'normal', 'size':14, 'ha':'left', 'va':'center'}
letters=['A','B','C','D','E','F','G','H','I','J','K','L','M','N']
tx=-105
ty=66
# vector specs
skip_nh=4
w=0.004
scalef=12
key_length=2
# h700 colormap
cmap=cm.RdYlBu_r
vmin=2900
vmax=3300
levels=np.linspace(vmin, vmax, 21)
norm=mpl.colors.BoundaryNorm(levels, cmap.N)
hlevels=np.arange(vmin-50, vmax+50, 25)
# swp700 colormap
dcmap=cm.RdBu_r
dlevels=np.linspace(-50, 50, 21)
dnorm=mpl.colors.BoundaryNorm(dlevels, dcmap.N)
# diff colormap
dcmap2=cmo.balance
dnorm2=mpl.colors.BoundaryNorm(np.linspace(-100, 100, 21), dcmap2.N)
dnorm3=mpl.colors.BoundaryNorm(np.linspace(-16, 16, 17), dcmap2.N)
# map specs
trans=ccrs.PlateCarree()
proj=ccrs.PlateCarree()
map_bnds=[225,290,10,65]
cols=[0,2]

fig, ax = plt.subplots(nrows=2, ncols=6, figsize=(35,10), layout='constrained', subplot_kw={'projection':proj})

for i,title in enumerate(['MERRA-2', 'CTRL', r'HI$\mathbf{_{gbl}}$', r'HI$\mathbf{_{mex}}$', r'HI$\mathbf{_{gbl}}$$-$CTRL', r'HI$\mathbf{_{mex}}$$-$CTRL']):
    ax[0,i].text(tx, ty, title, **text_kw)
    
#--- CLIMOS ---
for i,(key,topo_key) in enumerate(zip(jas_mean.keys(),topo.keys())):
    
    h700 = jas_mean[key]['h700']
    swp700 = jas_mean[key]['swp700']
    u700 = jas_mean[key]['u700']
    v700 = jas_mean[key]['v700']
    
    cf=ax[0,i].pcolormesh(h700.lon, h700.lat, h700, cmap=cmap, norm=norm, transform=trans)
    cf2=ax[1,i].pcolormesh(swp700.lon, swp700.lat, swp700, cmap=cmap, norm=norm, transform=trans)
    q=ax[0,i].quiver(u700.lon[::skip_nh], u700.lat[::skip_nh], u700[::skip_nh,::skip_nh], v700[::skip_nh,::skip_nh],
                     color='k', width=w, scale=scalef, scale_units='inches', units='height', transform=trans, zorder=100)
    ax[1,i].quiver(u700.lon[::skip_nh], u700.lat[::skip_nh], u700[::skip_nh,::skip_nh], v700[::skip_nh,::skip_nh],
                   color='k', width=w, scale=scalef, scale_units='inches', units='height', transform=trans, zorder=100)
    ax[0,i].quiverkey(q, .9, 1.05, key_length, rf'{key_length} m/s', labelcolor='k', labelpos='N', fontproperties={'size':8})


#--- FLOR CHANGE ---
ax[0,4].text(-165,40,' ')
for i,run in enumerate(['hitopo','hicam']):
    dh700 = model_diff[run]['h700']
    dswp700 = model_diff[run]['swp700']
    du700 = model_diff[run]['u700']
    dv700 = model_diff[run]['v700']
    
    cf3=ax[0,i+4].pcolormesh(dh700.lon, dh700.lat, dh700, cmap=dcmap2, norm=dnorm2, transform=trans)
    cf4=ax[1,i+4].pcolormesh(dswp700.lon, dswp700.lat, dswp700, cmap=dcmap2, norm=dnorm3, transform=trans)
    ax[0,i+4].quiver(du700.lon[::skip_nh], du700.lat[::skip_nh], du700[::skip_nh,::skip_nh], dv700[::skip_nh,::skip_nh],
                     color='k', width=w, scale=scalef, scale_units='inches', units='height', transform=trans, zorder=100)
    ax[1,i+4].quiver(du700.lon[::skip_nh], du700.lat[::skip_nh], du700[::skip_nh,::skip_nh], dv700[::skip_nh,::skip_nh],
                     color='k', width=w, scale=scalef, scale_units='inches', units='height', transform=trans, zorder=100)

# --- Formatting ---
for i, ax in enumerate(ax.flat): 
    # subplot labels
    ax.text(-136, ty+1, letters[i], **text_kw2)
    # map properties
    ax.coastlines(color='k', linewidth=1.5)
    ax.set_extent(map_bnds, crs=trans)
    gl=ax.gridlines(crs=trans, lw=.5, colors='black', alpha=1.0, linestyle='--', zorder=10, draw_labels=True)
    gl.bottom_labels=True; gl.left_labels=True; gl.top_labels=False; gl.right_labels=False
    gl.xformatter=LONGITUDE_FORMATTER
    gl.yformatter=LATITUDE_FORMATTER
    gl.xlabel_style={'color': 'black', 'weight': 'normal', 'size':14}
    gl.ylabel_style={'color': 'black', 'weight': 'normal', 'size':14}

# --- Colorbars ---
# h700
cax=fig.add_axes([.64, .55, 0.0125, 0.4])
cbar=fig.colorbar(cf, ticks=np.linspace(2900,3300,9), orientation='vertical', extend='both', cax=cax) 
cbar.set_label('700 hPa Geopotential Height [m]', size=14, rotation=270, labelpad=20, ha='center')
cbar.ax.tick_params(labelsize=14)
# swp700
cax=fig.add_axes([.64, .05, 0.0125, 0.4])
cbar=fig.colorbar(cf2, ticks=np.linspace(-50,50,11), orientation='vertical', extend='both', cax=cax) 
cbar.set_label('700 hPa Stationary Wave Pattern [m]', size=14, rotation=270, labelpad=20, ha='center')
cbar.ax.tick_params(labelsize=14)
# h700 change
cax=fig.add_axes([1.005, .55, 0.0125, 0.4])
cbar=fig.colorbar(cf3, ticks=np.linspace(-100,100,11), orientation='vertical', extend='both', cax=cax) 
cbar.set_label('$\Delta$ 700 hPa Geopotential Height [m]', size=14, rotation=270, labelpad=20, ha='center')
cbar.ax.tick_params(labelsize=14)
# swp700 change
cax=fig.add_axes([1.005, .05, 0.0125, 0.4])                                 
cbar=fig.colorbar(cf4, ticks=np.linspace(-16,16,9), orientation='vertical', extend='both', cax=cax) 
cbar.set_label('$\Delta$ 700 hPa Stationary Wave Pattern [m]', size=14, rotation=270, labelpad=20, ha='center')
cbar.ax.tick_params(labelsize=14)

#plt.savefig(f'figs/h700.swp.flor.pdf', transparent=False, bbox_inches='tight')
#plt.savefig(f'figs/h700.swp.flor.png', transparent=False, bbox_inches='tight')

In [ ]:
# --- Settings ---
# plot specs
lw=1
text_kw={'color':'k', 'weight':'bold', 'size':20, 'ha':'center', 'va':'bottom'}
text_kw2={'color':'k', 'weight':'bold', 'size':26, 'ha':'center', 'va':'center'}
text_kw3={'color':'k', 'weight':'normal', 'size':14, 'ha':'left', 'va':'center'}
letters=['A','B','C','D','E','F','G','H','I','J','K','L','M','N']
tx=-105
ty=66
# var specs
lon = jas_mean['pi']['ctrl']['h700'].lon
lat = jas_mean['pi']['ctrl']['h700'].lat
# topography contours
zlevels=np.linspace(1000,4000,8)
# vector specs
skip_nh=4
w=0.004
scalef=12
key_length=2
# swp700 colormap
dcmap=cmo.balance
dvmin=-20
dvmax=20
dlevels=np.linspace(dvmin, dvmax, 21)
dnorm=mpl.colors.BoundaryNorm(dlevels, dcmap.N)
# map specs
trans=ccrs.PlateCarree()
proj=ccrs.PlateCarree()
map_bnds=[225,290,10,65]
cols=[0,2]

fig, ax = plt.subplots(nrows=1, ncols=3, figsize=(15,10), layout='constrained', subplot_kw={'projection':proj})

for i,title in enumerate(['CTRL$-$MERRA2', r'HI$\mathbf{_{gbl}}$$-$MERRA2', r'HI$\mathbf{_{mex}}$$-$MERRA2']):
    ax[i].text(tx, ty, title, **text_kw)
    
#=== FLOR ===
for i,run in enumerate(['ctrl','hitopo','hicam']):
    cf2=ax[i].pcolormesh(lon, lat, obs_diff['pi'][run]['swp700'], cmap=dcmap, norm=dnorm, transform=trans)
    ax[i].quiver(lon[::skip_nh], lat[::skip_nh],
                 obs_diff['pi'][run]['u700'][::skip_nh,::skip_nh], obs_diff['pi'][run]['v700'][::skip_nh,::skip_nh],
                 color='k', width=w, scale=scalef, scale_units='inches', units='height', transform=trans, zorder=100)

for i, ax in enumerate(ax.flat): 
    # subplot labels
    ax.text(-136, ty+1, letters[i], **text_kw2)
    # map properties
    ax.coastlines(color='k', linewidth=1.5)
    ax.set_extent(map_bnds, crs=trans)
    gl=ax.gridlines(crs=trans, lw=.5, colors='black', alpha=1.0, linestyle='--', zorder=10, draw_labels=True)
    gl.bottom_labels=True; gl.left_labels=True; gl.top_labels=False; gl.right_labels=False
    gl.xformatter=LONGITUDE_FORMATTER
    gl.yformatter=LATITUDE_FORMATTER
    gl.xlabel_style={'color': 'black', 'weight': 'normal', 'size':14}
    gl.ylabel_style={'color': 'black', 'weight': 'normal', 'size':14}

# swp700
cax=fig.add_axes([1.01, .3, 0.0175, 0.4])
cbar=fig.colorbar(cf2, ticks=np.linspace(-50,50,11), orientation='vertical', extend='both', cax=cax) 
cbar.set_label('700 hPa Stationary Wave Pattern [m]', size=14, rotation=270, labelpad=20, ha='center')
cbar.ax.tick_params(labelsize=14)

#plt.savefig(f'figs/h700.swp.flor.pdf', transparent=False, bbox_inches='tight')
#plt.savefig(f'figs/h700.swp.flor.png', transparent=False, bbox_inches='tight')

In [ ]:
# --- Settings ---
# plot specs
lw=1
text_kw={'color':'k', 'weight':'bold', 'size':20, 'ha':'center', 'va':'bottom'}
text_kw2={'color':'k', 'weight':'bold', 'size':26, 'ha':'center', 'va':'center'}
text_kw3={'color':'k', 'weight':'normal', 'size':14, 'ha':'left', 'va':'center'}
letters=['A','B','C','D','E','F','G','H','I','J','K','L','M','N']
tx=-110
ty=40.5
# var specs
lon = jas_mean['pi']['ctrl']['h700'].lon
lat = jas_mean['pi']['ctrl']['h700'].lat
# topography contours
zlevels=np.linspace(1000,4000,8)
# vector specs
skip_nh=3
w=0.004
scalef=12
key_length=2
# swp700 colormap
dcmap=cmo.balance
dvmin=-20
dvmax=20
dlevels=np.linspace(dvmin, dvmax, 21)
dnorm=mpl.colors.BoundaryNorm(dlevels, dcmap.N)
# map specs
trans=ccrs.PlateCarree()
proj=ccrs.PlateCarree()
map_bnds=[230,270,10,40]
cols=[0,2]

fig, ax = plt.subplots(nrows=1, ncols=3, figsize=(15,10), layout='constrained', subplot_kw={'projection':proj})

for i,title in enumerate(['CTRL$-$MERRA2', r'HI$\mathbf{_{gbl}}$$-$MERRA2', r'HI$\mathbf{_{mex}}$$-$MERRA2']):
    ax[i].text(tx, ty, title, **text_kw)
    
#=== FLOR ===
for i,run in enumerate(['ctrl','hitopo','hicam']):
    cf2=ax[i].pcolormesh(lon, lat, obs_diff['pi'][run]['swp700'], cmap=dcmap, norm=dnorm, transform=trans)
    ax[i].quiver(lon[::skip_nh], lat[::skip_nh],
                 obs_diff['pi'][run]['u700'][::skip_nh,::skip_nh], obs_diff['pi'][run]['v700'][::skip_nh,::skip_nh],
                 color='k', width=w, scale=scalef, scale_units='inches', units='height', transform=trans, zorder=100)

for i, ax in enumerate(ax.flat): 
    # subplot labels
    ax.text(-130, ty+1, letters[i], **text_kw2)
    # map properties
    ax.coastlines(color='k', linewidth=1.5)
    ax.set_extent(map_bnds, crs=trans)
    gl=ax.gridlines(crs=trans, lw=.5, colors='black', alpha=1.0, linestyle='--', zorder=10, draw_labels=True)
    gl.bottom_labels=True; gl.left_labels=(i==0); gl.top_labels=False; gl.right_labels=False
    gl.xformatter=LONGITUDE_FORMATTER
    gl.yformatter=LATITUDE_FORMATTER
    gl.xlabel_style={'color': 'black', 'weight': 'normal', 'size':14}
    gl.ylabel_style={'color': 'black', 'weight': 'normal', 'size':14}

# swp700
cax=fig.add_axes([1.01, .3, 0.0175, 0.4])
cbar=fig.colorbar(cf2, ticks=np.linspace(-50,50,11), orientation='vertical', extend='both', cax=cax) 
cbar.set_label('700 hPa Stationary Wave Pattern [m]', size=14, rotation=270, labelpad=20, ha='center')
cbar.ax.tick_params(labelsize=14)

#plt.savefig(f'figs/h700.swp.flor.pdf', transparent=False, bbox_inches='tight')
#plt.savefig(f'figs/h700.swp.flor.png', transparent=False, bbox_inches='tight')